# Isolated environment smoke

Select a CPU runtime with Python 3.13. Synthetic computation only; no market data, Drive mount or final evaluation. CPU success does not validate GPU compatibility.


In [ ]:
_SETUP_VALIDATED = _CHECKOUT_READY = False
from pathlib import Path
import platform
import re
import subprocess
import sys

REPO_COMMIT = ""  # Full reviewed published commit.
REPO_DIR = Path("/content/yenibot_environment_smoke_repo")
ENV_DIR = Path("/content/yenibot_locked_env")
REPORT = Path("/content/yenibot_environment_smoke.json")
if sys.version_info[:2] != (3, 13) or platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("Choose Linux x86_64 / Python 3.13 before continuing")
if not re.fullmatch(r"[0-9a-f]{40}", REPO_COMMIT):
    raise ValueError("Set the reviewed full Git commit")
if REPO_DIR.exists() or ENV_DIR.exists() or REPORT.exists():
    raise FileExistsError("Use a fresh runtime or new paths; preserve existing evidence")
_SETUP_VALIDATED = True
RUN_GPU = False  # First run CPU; GPU requires a separate fresh GPU runtime.


In [ ]:
_CHECKOUT_READY = False
if not globals().get("_SETUP_VALIDATED", False):
    raise RuntimeError("Complete the first cell successfully")
subprocess.run(["git", "clone", "--no-checkout", "https://github.com/umutergul74/yeniBot.git", str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if actual != REPO_COMMIT:
    raise RuntimeError("Git identity mismatch")
_CHECKOUT_READY = True


In [ ]:
if not globals().get("_CHECKOUT_READY", False):
    raise RuntimeError("Complete checkout successfully before installation")
from datetime import datetime, timezone
LOG = Path("/content") / ("yenibot_setup_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f") + ".log")
command = [sys.executable, "-u", str(REPO_DIR / "scripts/prepare_environment.py"), "--directory", str(ENV_DIR), "--report", str(REPORT)]
if RUN_GPU:
    command.append("--gpu")
with LOG.open("x", encoding="utf-8") as log:
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        code = process.wait()
print("Full setup log:", LOG)
if code:
    raise RuntimeError(f"Setup failed (exit {code}). Share the ERROR/Traceback above and {LOG}; preserve this runtime.")
print(REPORT.read_text(encoding="utf-8"))


Preserve the report before the runtime expires. Existing research notebooks do not automatically use this environment. Stop here; do not start final evaluation as a smoke test.
